# Stage 1 — Data Pipeline

This notebook narrates and demonstrates the full data preprocessing pipeline for the OceanEmbed PoC. It imports from the existing `src/` modules and calls `scripts/prepare_data.py` to process the raw NetCDF data into PyTorch-ready training sets.

### Input Channels

| Channel | Name | Source Dataset | Resolution | Physical Meaning |
| :--- | :--- | :--- | :--- | :--- |
| `sst` | Sea Surface Temperature | OSTIA | 0.05° | Surface temperature |
| `grad_sst` | SST Gradient | OSTIA | 0.05° | Magnitude of spatial SST gradient |
| `sla` | Sea Level Anomaly | CMEMS | 0.25° | Deviation of sea surface height |
| `grad_sla` | SLA Gradient | CMEMS | 0.25° | Magnitude of spatial SLA gradient |
| `sss` | Sea Surface Salinity | SMAP | 0.25° | Surface salinity |
| `wsc` | Wind Stress Curl | CCMP | 0.25° | Curl of wind stress (derived from u/v winds) |
| `u_curr` | Zonal Surface Current | OSCAR | 0.25° | Eastward surface current velocity |
| `v_curr` | Meridional Surface Current | OSCAR | 0.25° | Northward surface current velocity |

### Target Depth Levels

The 15 target depth levels (in meters) are: `[0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000]`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import yaml
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xarray as xr

cfg = yaml.safe_load(open('../configs/poc.yaml'))
print('Domain:', cfg['data']['lat_min'], '–', cfg['data']['lat_max'], '°N,', cfg['data']['lon_min'], '–', cfg['data']['lon_max'], '°E')
print('Grid resolution:', cfg['data']['grid_res'], '° →', int((cfg['data']['lat_max']-cfg['data']['lat_min'])/cfg['data']['grid_res']), '×', int((cfg['data']['lon_max']-cfg['data']['lon_min'])/cfg['data']['grid_res']), 'cells')
print('Input channels:', cfg['channels']['names'])
print('Depth levels:', cfg['standard_depths'])
print('Temporal window:', cfg['data']['temporal_window'], 'days →', cfg['data']['temporal_window'] * len(cfg['channels']['names']), 'input channels')
print('Splits:')
print(' Train:', cfg['data']['train_start'], '–', cfg['data']['train_end'])
print(' Val: ', cfg['data']['val_start'], '–', cfg['data']['val_end'])
print(' Test: ', cfg['data']['test_start'], '–', cfg['data']['test_end'])

### Preprocessing Steps

The data preparation pipeline performs the following steps:
1. **Regridding**: Regrid all source datasets to the 0.5° target grid via bilinear interpolation.
2. **Derived Features**: Compute derived features (∇SST, ∇SLA as log-magnitude, Wind Stress Curl from CCMP winds).
3. **Normalization**: Z-score normalize each channel globally.
4. **Temporal Stacking**: Stack a 3-day temporal window for inputs.

*Note: Raw wind components are NEVER used — only WSC.*

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../scripts/prepare_data.py', '--splits', 'train', 'val', '--config', '../configs/poc.yaml'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

### Processed Output Inspection

The preprocessing script generates standardized NetCDF files in `outputs/processed/`. The dataset is split into training, validation, and testing sets, with corresponding input and target files. Let's inspect the shapes of the processed data.

In [ ]:
ds_in = xr.open_dataset('../outputs/processed/train/inputs_2015.nc')
ds_tg = xr.open_dataset('../outputs/processed/train/targets_2015.nc')
print('Input dataset:')
print(ds_in)
print('\nTarget dataset:')
print(ds_tg)
print('\nInput shape: (time, channels, lat, lon) =', ds_in['inputs'].shape)
print('Target shape: (time, depths, lat, lon) =', ds_tg['targets'].shape)

Let's visualise one day's worth of surface inputs — all 8 channels across the North Indian Ocean domain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ds = xr.open_dataset('../outputs/processed/train/inputs_2015.nc')
day_idx = 100  # some day in 2015
channels = cfg['channels']['names']
lats = ds.lat.values
lons = ds.lon.values

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
cmaps = ['RdYlBu_r', 'viridis', 'RdBu_r', 'viridis', 'RdYlBu_r', 'PuOr', 'RdBu_r', 'RdBu_r']
titles = ['SST (°C normalized)', '|∇SST| log-mag', 'SLA (m normalized)', '|∇SLA| log-mag',
          'SSS (psu normalized)', 'Wind Stress Curl', 'U_curr (m/s norm)', 'V_curr (m/s norm)']

for i, (ax, ch, cmap, title) in enumerate(zip(axes, channels, cmaps, titles)):
    data = ds['inputs'].values[day_idx, i]
    vmax = np.nanpercentile(np.abs(data), 98)
    im = ax.pcolormesh(lons, lats, data, cmap=cmap, vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Lon (°E)', fontsize=7)
    ax.set_ylabel('Lat (°N)', fontsize=7)
    ax.tick_params(labelsize=7)

fig.suptitle(f'OceanEmbed PoC — 8 Input Channels\nDay index {day_idx} of training set (2015)', 
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/input_channels_sample.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to outputs/figures/input_channels_sample.png')

Now let's look at the GLORYS training targets — temperature at 5 representative depths for the same day.

In [ ]:
ds_tg = xr.open_dataset('../outputs/processed/train/targets_2015.nc')
depths = cfg['standard_depths']
show_depths = [0, 4, 7, 11, 14]  # 0m, 30m, 100m, 300m, 1000m
show_labels = [f'{depths[i]} m' for i in show_depths]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, di, label in zip(axes, show_depths, show_labels):
    data = ds_tg['targets'].values[day_idx, di]
    # denorm is not available easily here — show normalized
    im = ax.pcolormesh(lons, lats, data, cmap='RdYlBu_r', vmin=-3, vmax=3)
    plt.colorbar(im, ax=ax, shrink=0.85, label='Normalized T')
    ax.set_title(f'Target: {label}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Lon (°E)', fontsize=7)
    ax.set_ylabel('Lat (°N)', fontsize=7)
    ax.tick_params(labelsize=7)

fig.suptitle('GLORYS Training Targets — Temperature at 5 Depths (normalized)', 
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/target_depths_sample.png', dpi=120, bbox_inches='tight')
plt.show()

Normalisation statistics computed from 2015-2017 training data:

In [ ]:
import json
stats = json.load(open('../outputs/processed/norm_stats.json'))
print('Input channel normalisation stats (Z-score, training 2015-2017):')
print(f'{"Channel":<12} {"Mean":>12} {"Std":>12}')
print('-' * 38)
for ch in cfg['channels']['names']:
    s = stats['inputs'][ch]
    print(f'{ch:<12} {s["mean"]:>12.4f} {s["std"]:>12.4f}')
print()
print('Target depth normalisation stats (first 5 depths):')
print(f'{"Depth":<10} {"Mean":>10} {"Std":>10}')
print('-' * 32)
for d in cfg['standard_depths'][:5]:
    k = f'd{d}m'
    s = stats['targets'][k]
    print(f'{k:<10} {s["mean"]:>10.4f} {s["std"]:>10.4f}')

### Pipeline Summary

This pipeline successfully converts raw cached satellite and reanalysis data (approx. 250GB) into standardized, normalized PyTorch-ready tensors. The processing involves regridding to a common 0.5° resolution grid, extracting relevant features like spatial gradients and wind stress curl, z-score normalizing the data using statistics from the training set (2015-2017), and stacking inputs over a 3-day temporal window to provide dynamic context for the target temperature profiles.